# AskJack — Test du RAG (routage + retrieval + génération)

Teste `src/rag.py`, le même module que celui de l'app Streamlit.

Pipeline : **get_intent** (JSON : intent, doc_types, project_ids, skills) → **retrieve** (pré-filtre → recherche → complément hors filtre) → **génération**.

## 0. Setup

In [ ]:
import os, sys, json
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

%load_ext autoreload
%autoreload 2
from src import rag

CFG = rag.load_config(ROOT)
vs  = rag.load_vectorstore(CFG, ROOT)
llm = rag.get_llm(CFG)

print("mode :", CFG["app"]["mode"], "| debug :", CFG["app"]["debug"],
      "| LLM :", CFG["models"]["llm"], "| top_k :", CFG["retrieval"]["top_k"],
      "| index :", vs.index.ntotal, "vecteurs")

In [ ]:
print("DOC TYPES :")
for k, v in rag.source_catalog(CFG).items():
    print(f"  {k:14} {v}")
print("PROJETS :")
for k, v in rag.project_catalog(vs).items():
    print(f"  P{k:<3} {v}")

## 1. Helpers d'affichage

In [ ]:
def show(res: dict, debug: bool | None = None):
    debug = CFG["app"]["debug"] if debug is None else debug
    print(f"[{res['mode']}] {res['answer']}")
    if not debug:
        return
    if res["intent"]:
        print("\nintent  :", {k: v for k, v in res["intent"].items() if k != "usage"})
    print("filtres :", res["filters"])
    u = res["usage"] or {}
    print(f"tokens  -> in: {u.get('input_tokens')} | out: {u.get('output_tokens')}")
    if res["sources"]:
        display(pd.DataFrame(res["sources"])[["rank", "origin", "distance", "doc_type", "project_id", "chunk_id"]])

def show_prompt(res: dict):
    for m in res["messages"]:
        print(f"===== {m.type.upper()} =====\n{m.content}\n")

def ask(q, **kw):
    res = rag.ask(q, CFG, vs, llm, **kw)
    show(res)
    return res

## 2. Le routeur seul (1 petit appel par question)

In [ ]:
for q in [
    "Qu'a pensé l'évaluateur du projet 5 ?",
    "Quels packages Python a-t-il utilisés pour les projets de tuype RAG ?",
    "Présente-moi son parcours pro.",
    "Quelle stack MLOps maîtrise-t-il ?",
    "Data Scientist H/F — Python, FastAPI, Docker, MLflow, une expérience RAG est un plus.",
    "Salut pozzi c'est koi la meteo demain ?",
    "La recette pick pick please?",
    "Yop !",
]:
    it = rag.get_intent(q, CFG, vs, llm)
    print(f"{q[:60]:60} -> { {k: v for k, v in it.items() if k != 'usage'} }")

## 3. Une question complète, avec le prompt final

In [ ]:
res = ask("Quels projets impliquent du déploiement d'API, et avec quelle stack ?")

In [ ]:
show_prompt(res)

## 4. Comparaisons

In [ ]:
q = "Qu'a pensé l'évaluateur du projet 5 ?"
_ = ask(q)                      # avec routeur
_ = ask(q, use_router=False)    # sans routeur (recherche sur tout l'index)

In [ ]:
print([l for l in CFG["prompts"]["router"].splitlines() if "project_ids" in l])

In [ ]:
#q = "Quelles compétences MLOps ont été acquises pendant la formation ?"
#q = "Quels projets utilisent des compétences RAG?"
q = "Salut pozzi, quel météo pour demain?"
_ = ask(q, mode="recrut")
#_ = ask(q, mode="candidat")

In [ ]:
# Filtre manuel : court-circuite le routeur
_ = ask("Qu'a pensé l'évaluateur de ce projet 5?")

## 5. Batterie de questions type recruteur

In [ ]:
QUESTIONS = [
    "Présente-moi rapidement le parcours du candidat.",
    "Quels langages et librairies Python maîtrise-t-il ?",
    "A-t-il déjà travaillé sur du NLP ou des LLM ?",
    "Quels modèles de machine learning a-t-il entraînés ?",
    "Comment gère-t-il le suivi des expériences (MLflow, etc.) ?",
    "A-t-il de l'expérience avec Docker ou le cloud ?",
    "Quels ont été les points faibles relevés par les évaluateurs ?",
    "A-t-il fait du deep learning / de la computer vision ?",
    "Sait-il faire du Kubernetes ?",          # hors corpus a priori -> ne doit rien inventer
]
results = {q: ask(q, mode="recrut") for q in QUESTIONS}

## 6. Mode candidat — réponse à une annonce (routeur -> intent annonce -> 1 recherche par compétence)

In [ ]:
ANNONCE = """
Data Scientist H/F — Nantes
Vous concevez et déployez des modèles de ML en production (Python, scikit-learn, FastAPI, Docker).
Une expérience sur des sujets NLP / LLM (RAG) est un plus. Connaissance de MLflow et CI/CD appréciée.
"""
res_annonce = ask(f"Réponds à l'annonce suivante :\n{ANNONCE}", mode="candidat")